# 01. Подготовка данных

Локальный pipeline скачивает четыре публичных архива, материализует только необходимые изображения и кропы и создаёт splits. NII не используется. Все операции записи выключены по умолчанию.

In [ ]:
from pathlib import Path
import json
from core.notebook_runtime import bootstrap_notebook, describe_runtime, run_guarded

GPU_INDEX = 0
context = bootstrap_notebook(gpu_index=GPU_INDEX)
PROJECT_ROOT = context.project_root
CONFIG = PROJECT_ROOT / 'configs/data.yaml'
DATA_ROOT = PROJECT_ROOT / 'data'
RUN_ENVIRONMENT_CHECK = False
RUN_DRY_RUN = False
RUN_DOWNLOAD_BUILD_SPLIT = False
describe_runtime(context)

## Конфигурация источников

In [ ]:
from core.config_loader import load_config
config = load_config(CONFIG)
print('Sources:', sorted(config['sources']))
assert 'nii' not in {name.lower() for name in config['sources']}
print('Detector split:', config['splits']['detector'])
print('Classifier validation fraction:', config['splits']['classifier_validation'])
print('Gunduz gallery fraction:', config['splits']['benchmark_gallery'])

## Проверка локального окружения
Команда только сообщает версию CUDA, GPU и свободное место; ничего не скачивает.

In [ ]:
run_guarded(
    context,
    context.module_command('scripts.check_environment', '--data-root', 'data', '--minimum-free-gb', '100', '--gpu-index', '0', '--require-jupyter'),
    enabled=RUN_ENVIRONMENT_CHECK,
    label='environment-check',
)

## План и сборка
Сначала выполните dry-run. Для реальной загрузки и материализации отдельно включите последний флаг.

In [ ]:
base_command = context.module_command('scripts.prepare_data', 'all', '--config', str(CONFIG))
run_guarded(context, [*base_command, '--dry-run'], enabled=RUN_DRY_RUN, label='data-dry-run')
if RUN_DOWNLOAD_BUILD_SPLIT:
    assert not (DATA_ROOT / 'datasetDiatom').exists(), 'Готовый dataset не перезаписывается: выберите новый data_root'
run_guarded(context, base_command, enabled=RUN_DOWNLOAD_BUILD_SPLIT, label='data-build')

## Аудит результата

In [ ]:
audit_files = sorted(DATA_ROOT.glob('**/audit.json'))
for path in audit_files:
    print('\n', path.relative_to(PROJECT_ROOT))
    print(json.dumps(json.loads(path.read_text(encoding='utf-8')), ensure_ascii=False, indent=2))
if not audit_files:
    print('Audit files появятся после сборки splits')